# Feature Engineering Using Scikit-Learn

### Do You Want to Build a Snowman?

Prepare the inputs for a model that predicts snowman height.

<img src="./assets/olaf.jpeg" width="300" alt="Snowman illustration">

This deliberately tiny, fictional dataset is a **preprocessing exercise**, not evidence that lunch or dinner predicts snowman height. Treat all rows as a supplied training sample; do not report a model generalization score from these 14 examples.

**Business connection:** in a housing dataset, the same steps could impute missing floor areas, scale numerical measurements, and encode property types before predicting price. The workflow transfers; the useful features and preprocessing choices depend on the task.

#### Load Packages

In [1]:
import numpy as np
import pandas as pd

#### Load Data

In [2]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data).fillna(np.nan)
df_train

,temp,lunch,dinner,precipitation,height_snowman_cm
0,-3.0,soup,pizza,yes,100
1,5.0,sandwich,pizza,no,0
2,0.0,soup,noodles,yes,75
3,7.0,burger,NaN,yes,0
4,3.0,sandwich,fishsticks,yes,20
5,-1.0,soup,pizza,yes,25
6,1.0,cereal,NaN,no,0
7,NaN,salad,fishsticks,yes,35
8,-6.0,sandwich,noodles,yes,170
9,3.0,burger,pizza,no,0


## Exercise

1. Separate `df_train` into `X_train` and `y_train`; the target is `height_snowman_cm`.
2. Identify numeric, nominal, and binary inputs, then inspect missing values.
3. Choose and justify an imputation strategy for each incomplete input. Row order has no time meaning here, so interpolation between neighboring rows is not justified.
4. Choose suitable encodings and a scaler. Explain which columns need each transformation and avoid inventing an order for nominal categories.
5. Import the scikit-learn tools you choose, apply them, and combine the transformed columns into a DataFrame.
6. Produce `X_train_fe` with numerical values, no missing values, the same row count and index as `X_train`, and no target column.

Fit on training data only. To transform new rows later, reuse the fitted objects instead of recomputing statistics.

In [17]:
target = "height_snowman_cm"

X_train = df_train.drop(target, axis=1)
y_train = df_train[target]

df_train.info()

#inspect missing values
df_train.isnull().sum()

# show only the rows that contain at least one missing value
df_train[df_train.isnull().any(axis=1)]

# 3. imputation (fit on training data only)
from sklearn.impute import SimpleImputer

# temp is numeric -> mean
temp_imputer = SimpleImputer(strategy="mean")
# dinner is categorical -> a mean is impossible, so use the most frequent value
dinner_imputer = SimpleImputer(strategy="most_frequent")

X_train_imputed = X_train.copy()
X_train_imputed[["temp"]] = temp_imputer.fit_transform(X_train[["temp"]])
X_train_imputed[["dinner"]] = dinner_imputer.fit_transform(X_train[["dinner"]])

X_train_imputed.isnull().sum()
df_train
X_train

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   temp               12 non-null     float64
 1   lunch              14 non-null     str    
 2   dinner             11 non-null     str    
 3   precipitation      14 non-null     str    
 4   height_snowman_cm  14 non-null     int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 692.0 bytes


,temp,lunch,dinner,precipitation
0,-3.0,soup,pizza,yes
1,5.0,sandwich,pizza,no
2,0.0,soup,noodles,yes
3,7.0,burger,NaN,yes
4,3.0,sandwich,fishsticks,yes
5,-1.0,soup,pizza,yes
6,1.0,cereal,NaN,no
7,NaN,salad,fishsticks,yes
8,-6.0,sandwich,noodles,yes
9,3.0,burger,pizza,no


In [ ]:
# New inputs only: the fitted transformers must not learn from these rows.
X_new = pd.DataFrame(
    {
        "temp": [np.nan, 2.0],
        "lunch": ["soup", "wrap"],
        "dinner": [np.nan, "pizza"],
        "precipitation": ["yes", "no"],
    },
    index=["known", "new_category"],
)
X_new

### Check Your Understanding

1. Why should you use `transform`, not `fit_transform`, on the test set?
2. Why could encoding meals as 0, 1, 2 mislead a linear or distance-based model?
3. Does `RobustScaler` remove extreme observations? Explain briefly.

Compare your reasoning with the short answers at the end of the solution.